# Genome Garden in Colab

Watch evolution happen in your browser, with nothing to install. Python runs on Google's computer and the world appears right here in the notebook.

**To start:** choose **Runtime → Run all** in the menu. If Colab warns that the notebook was not written by Google, click **Run anyway**. After a few seconds the world appears under step 2. You need to be signed in to a Google account.

## 1. Download Genome Garden
This takes a few seconds.

In [ ]:
import os, subprocess, sys

BRANCH = "main"
HOME = "/content/genome-garden"
if not os.path.exists(HOME):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, "https://github.com/sbbis25/genome-garden", HOME], check=True)
else:    # already downloaded in this session: fetch the latest of BRANCH, keeping your workshop.py edits
    git = lambda *a: subprocess.run(["git", "-C", HOME] + list(a)).returncode
    if git("fetch", "-q", "--depth", "1", "origin", BRANCH) or git("checkout", "-q", "-B", BRANCH, "FETCH_HEAD"):
        print("Could not download updates, so the copy already here is used.")
os.chdir(HOME)
sys.path.insert(0, HOME)

# python keeps the first garden it loaded, so an update needs a fresh session to take effect
commit = subprocess.run(["git", "-C", HOME, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if "garden" not in sys.modules:
    os.environ["GG_LOADED"] = commit
elif os.environ.get("GG_LOADED") != commit:
    raise SystemExit("Genome Garden was updated. Choose Runtime → Restart session, then Runtime → Run all.")
print("Ready.")

## 2. Start the world
The garden appears below this cell and keeps running while this notebook is open. It only works inside the notebook, not in a tab of its own.

In [ ]:
import threading
from http.server import ThreadingHTTPServer
from google.colab import output
from garden.server import Sim, make_handler
from garden.world import World

if "garden_port" not in globals():    # start it once; running this cell again just shows it again
    sim = Sim(World(HOME + "/workshop.py"))
    sim.start()
    ThreadingHTTPServer.daemon_threads = True
    for garden_port in range(8765, 8790):    # the first free port
        try:
            httpd = ThreadingHTTPServer(("127.0.0.1", garden_port), make_handler(sim))
            break
        except OSError:
            pass
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
output.serve_kernel_port_as_iframe(garden_port, height=820)

## 3. Write your own rules
Run this cell to open `workshop.py`, the one file you edit, in an editor on the right. Uncomment an example (delete the `# ` at the start of each line), then save with **Ctrl+S** (**Cmd+S** on a Mac). The running world picks up your change within a second. Mistakes show up in the garden's **Code** tab with the line number, and the world keeps running.

You do not need to run cell 2 again after you edit.

In [ ]:
from google.colab import files

files.view(HOME + "/workshop.py")

## 4. Keep your code
Your `workshop.py` lives on Google's computer and is deleted when the session ends. Before you leave, run this cell (click it, then press the ▶ button on its left) to download a copy. It does nothing until you have changed the file.

In [ ]:
import subprocess
from google.colab import files

# only once the file has changed, so "run all" does not download an untouched copy
if subprocess.run(["git", "-C", HOME, "diff", "--quiet", "workshop.py"]).returncode:
    files.download(HOME + "/workshop.py")
else:
    print("You have not changed workshop.py yet, so there is nothing to download.")

## If something goes wrong
- **The garden stopped updating:** Colab closes sessions that sit idle for a while, or when you close the notebook. Choose **Runtime → Run all**.
- **A blank box or an error under step 2:** choose **Runtime → Disconnect and delete runtime**, then **Runtime → Run all**.
- **It feels choppy:** every frame travels through Google's servers, so it is smoother when run on your own laptop with `python run.py`.

## Connection test
Only for when the world will not show up under step 2: an organizer may ask you to run this. It starts a plain test server (A) and two test copies of the garden (B and C), and shows each one below. Then **run this cell again** to list what each one received, and send the whole output to the organizer.

In [ ]:
import subprocess, sys, threading, time, urllib.request
from http.server import ThreadingHTTPServer
from google.colab import output
from garden.server import Sim, make_handler
from garden.world import World

def garden_running():
    try:
        return urllib.request.urlopen("http://127.0.0.1:%d/" % garden_port, timeout=2).status == 200
    except Exception:
        return False

if "diag" not in globals() and "garden_port" in globals() and garden_running():
    print("The garden is running, so there is nothing to test.")
elif "diag" not in globals():
    diag = {"B": [], "C": []}

    def logging_garden(host, port, seen):
        sim = Sim(World(HOME + "/workshop.py")); sim.start()

        class Logged(make_handler(sim)):
            def log_message(self, fmt, *args):
                seen.append("%s %s" % (self.headers.get("Host"), fmt % args))

        httpd = ThreadingHTTPServer((host, port), Logged)
        threading.Thread(target=httpd.serve_forever, daemon=True).start()

    # a: python's own test server; b: garden open to this machine only; c: garden open to all
    subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--bind", "0.0.0.0"], cwd="/content",
                     stdout=open("/content/diag_a.log", "w"), stderr=subprocess.STDOUT)
    logging_garden("127.0.0.1", 8770, diag["B"])
    logging_garden("0.0.0.0", 8780, diag["C"])
    time.sleep(1)
    for name, p in (("A", 8000), ("B", 8770), ("C", 8780)):
        try:
            print(name, "answers inside colab:", urllib.request.urlopen("http://127.0.0.1:%d/" % p).status)
        except Exception as e:
            print(name, "does not answer inside colab:", e)
    print("browser:", output.eval_js("navigator.userAgent"))
    for name, p in (("A: test server", 8000), ("B: test garden, this machine only", 8770), ("C: test garden, open to all", 8780)):
        print("\n" + name)
        output.serve_kernel_port_as_iframe(p, height=300)
else:
    print("A received:"); print(open("/content/diag_a.log").read() or "  nothing")
    for name in ("B", "C"):
        print(name, "received:"); print("\n".join(diag[name][-15:]) or "  nothing")